# Task 2 - Financial News Impact Analysis: QLoRA Fine-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya/blob/main/task2_genai/task2_news_impact_finetuning.ipynb)

Fine-tune **Qwen2.5-1.5B-Instruct** with QLoRA (4-bit NF4) so it turns a financial news item into a structured impact
analysis (event type, entities, direction, magnitude, rationale) more consistently and with fewer hallucinations than
the same model with a prompt alone. Problem statement, dataset construction and limitations:
[`task2_genai/README.md`](README.md).

| Part | In this notebook |
|---|---|
| **2A** Dataset | sizes, diversity charts, chat-template rendering, teacher prompt (data was generated by `generate_data.py`) |
| **2B** Fine-tuning | 4-bit NF4 + LoRA, every hyperparameter justified, train/val loss per epoch, merged model pushed to the Hub |
| **2C** Evaluation | ROUGE-L, BERTScore, field accuracy and LLM-as-judge for base vs fine-tuned; manual hallucination review |

**Run on Colab:** *Runtime → Change runtime type → T4 GPU*. Add two Colab Secrets (key icon, left sidebar):
`HF_TOKEN` (a Hugging Face token with **write** access, used to push the model) and optionally `GROQ_API_KEY`
(only for the LLM-as-judge in 2C). Then *Runtime → Run all*. No keys are stored in this notebook.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/indeewara/CDAZZDEV-MLE-IndeewaraJayasuriya.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd):
    # run a command and SHOW its output if it fails - Colab otherwise hides subprocess output
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"command failed ({r.returncode}): {' '.join(cmd[:6])} ...")
    return r

if IN_COLAB:
    if not os.path.exists("CDAZZDEV-MLE-IndeewaraJayasuriya"):
        sh(["git", "clone", "-q", REPO_URL])
    else:  # re-running in the same session: fetch the latest code instead of using a stale copy
        sh(["git", "-C", "CDAZZDEV-MLE-IndeewaraJayasuriya", "pull", "-q"])
        print("repo updated - if code changed, use Runtime > Restart session, then Run all (Python caches imported modules)")
    os.chdir("CDAZZDEV-MLE-IndeewaraJayasuriya/task2_genai")
    # Pinned versions: these libraries change their APIs between releases (e.g. transformers 5 removed
    # warmup_ratio, TRL renamed max_seq_length). torch is Colab's preinstalled CUDA build.
    sh([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.18.0", "trl==1.14.1", "peft==0.21.2", "accelerate==1.15.0",
                    "datasets==5.1.0", "bitsandbytes>=0.45", "pydantic", "python-dotenv", "matplotlib",
                    "groq", "rouge-score", "bert-score"])
    # Colab preinstalls torchao 0.10, which PEFT 0.21 rejects (it requires >= 0.16) when loading the adapter
    # for the merge. This project does not use torchao, so remove it rather than upgrade it.
    sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    print("libraries installed")
    from google.colab import userdata
    for key in ("HF_TOKEN", "GROQ_API_KEY"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:  # secret not set - only HF_TOKEN is required, for the push
            print(f"Colab secret {key} not set")

sys.path.insert(0, os.getcwd())
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("working dir: .../" + os.path.basename(os.getcwd()), "| Colab:", IN_COLAB)
print("code version:", commit)   # compare with the latest commit on GitHub if results look stale

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1e9:.1f} GB")
else:
    print("No GPU - switch the runtime to T4 GPU; 4-bit training needs CUDA.")

---
# Task 2A - Dataset

Generated by `generate_data.py` (teacher `openai/gpt-oss-120b`, blind labeller `openai/gpt-oss-20b`, both via the Groq
free tier) and committed to the repo, so this notebook does not need to call the teacher again. `dataset_report.py`
re-validates every record, measures token lengths with Qwen's tokenizer, and redraws the diversity charts from the
files.

In [ ]:
import json
import dataset_report

report = dataset_report.main()
print(json.dumps(report["summary"], indent=2))

In [ ]:
from IPython.display import Image, display
display(Image("outputs/dataset_diversity.png"))

In [ ]:
div = json.load(open("data/diversity_report.json"))
print("planned specs:", div["planned_specs"], "| kept:", div["total"], "| rejected:", div["rejected"])
print("automatic fixes:", div["automatic_fixes"])
print("distinct word bigrams:", div["distinct_2"], "| nearest-neighbour similarity (word level):", div["nearest_neighbour_similarity"])
print("top keywords:", [w for w, _ in div["top_keywords"][:20]])

One training record rendered in **Qwen2.5's chat template** (system, user and assistant turns):

In [ ]:
print(report["rendered_example"])

**Full system prompt used for data generation** (the teacher). All prompts, including the blind labeller's, are in
[`TEACHER_PROMPTS.md`](TEACHER_PROMPTS.md), written verbatim from `news_prompts.py`.

In [ ]:
from news_prompts import GENERATION_SYSTEM, GENERATION_USER
print(GENERATION_SYSTEM)
print("\n--- user prompt template ---\n" + GENERATION_USER)

---
# Task 2B - QLoRA Fine-Tuning

All settings live in `finetune.py` as named constants, each with its reason beside it; this table repeats them.

### Hyperparameters and why

| Parameter | Value | Why |
|---|---|---|
| Base model | Qwen2.5-1.5B-Instruct | Instruction-tuned, so fine-tuning teaches the task rather than basic instruction following. Fits a free T4 in 4-bit with room for batch 4 × 1024 tokens. Apache-2.0, not gated. Different family from the teacher (OpenAI gpt-oss). |
| Quantization | 4-bit **NF4**, double quantization | NF4 is the optimal 4-bit grid for normally distributed weights (QLoRA paper); double quantization also compresses the quantization constants (~0.4 bits/param) at no quality cost. |
| Compute dtype | float16 | The T4 (Turing) has no bfloat16 support. |
| 4-bit storage dtype | uint8 (set explicitly) | How packed 4-bit weights are stored; it only matters for multi-GPU (FSDP) sharding, which a single T4 does not use. |
| **LoRA r** | **16** | Narrow task (fixed taxonomy, JSON format, short rationale) and 149 examples: r=8 risks underfitting 11-way classification plus generation; r=64 adds capacity we cannot fill and overfits faster. ~18M trainable params (~1.2%). |
| **LoRA alpha** | **32** | alpha/r = 2, the usual scaling at r=16; keeps update size stable so the learning rate can be reasoned about independently of r. |
| LoRA dropout | 0.05 | Light regularisation for a small dataset; higher slows learning in a 30-step run. |
| **Target modules** | **q, k, v, o, gate, up, down projections** | All linear layers. The QLoRA paper found adapting every linear layer is needed to match full fine-tuning; format and taxonomy knowledge is not confined to attention. |
| LoRA bias | none | Biases stay frozen (standard); the merged model keeps the base architecture. |
| LoRA layers | all 28 transformer layers | Format and labelling behaviour is spread through the depth of the network, not a few layers. |
| Modules to save | none - embeddings and LM head frozen | No tokens are added, so the vocabulary and output layer need no training. |
| rsLoRA | off | Rank-stabilised scaling (alpha/√r) helps at high ranks (64+); at r=16 the standard alpha/r scaling is well-behaved and comparable to the QLoRA paper. |
| DoRA | off | Adds a learned magnitude vector per layer: more memory and ~2× slower steps on a T4, aimed at harder tasks than format + taxonomy learning. |
| **Learning rate** | **2e-4** | QLoRA paper value for small models. LoRA's B matrix starts at zero, so it needs a higher rate than full fine-tuning (~1e-5) to move meaningfully in 30 steps. |
| **LR scheduler** | **cosine**, 10% warm-up | Warm-up (~3 steps) lets Adam's moment estimates settle; cosine decays to ~0 so the last epoch makes small updates, limiting late overfitting. |
| **Epochs** | **3** | 149 / 16 ≈ 10 optimizer steps per epoch. One epoch (10 steps) is too few for an 11-type taxonomy; beyond 3 the model memorises the teacher's phrasing. Validation loss per epoch is the check. |
| **Batch size** | **4** per device | Largest that fits a T4 at ~800 tokens per sequence with 4-bit weights and gradient checkpointing. |
| **Gradient accumulation** | **4** | Effective batch 16: smoother gradients than 4 while keeping 10 steps per epoch. |
| **Max sequence length** | **1024** | Longest sequence is 810 tokens (measured in 2A), so nothing is truncated; larger only wastes memory. |
| Loss | assistant tokens only | The ~600-token system prompt is identical in every example; counting it would make loss mostly "memorise the prompt". Verified: 81 of 756 tokens carry loss in a typical example. |
| Optimizer | paged AdamW 8-bit | QLoRA's paged optimizer spills state to CPU RAM on memory spikes instead of crashing; 8-bit states cut optimizer memory ~4×. |
| Adam betas / epsilon | 0.9, 0.999 / 1e-8 (set explicitly) | The standard AdamW values used by the QLoRA paper; Adam's bias correction makes beta2 = 0.999 behave sensibly even over 30 steps. |
| Label smoothing | 0.0 | Targets are exact labels from a fixed set and must form valid JSON; smoothing would reward spreading probability onto other tokens. |
| NEFTune noise | off | Embedding noise makes open-ended chat more varied; this task needs exact, consistent structured output. |
| torch.compile | off | Compilation overhead exceeds its gain over 30 steps, and is fragile with 4-bit layers. |
| Drop last batch | no | 149 is not a multiple of 4; the last short batch is kept so every example is seen each epoch. |
| Weight decay | 0.0 | LoRA dropout already regularises; decaying zero-initialised adapters over 30 steps mostly fights the learning signal. |
| Max grad norm | 0.3 | QLoRA paper value; clips occasional large gradients caused by 4-bit noise. |
| Gradient checkpointing | on (non-reentrant) | ~40% less activation memory for ~20% more time; non-reentrant mode is required with PEFT adapters. |
| Mixed precision | fp16 | Matches the compute dtype on the T4. |
| Packing | off | Each example is an independent news item; packing would put unrelated items in one sequence. |
| Eval / save / logging | every epoch | The assessment asks for train and validation loss per epoch. |
| Best model | lowest validation loss is restored at the end | Saves the best epoch even if a later one is worse. |
| LoRA initialisation | PEFT default (A random, B zero) | The adapter starts as an exact no-op, so training begins from the base model's behaviour. |
| Seed | 42 | Reproducible shuffling, LoRA initialisation and dropout. |

Every setting above is set **explicitly** in `finetune.py` (none relies on a silent library default). The remaining
`TrainingArguments` are bookkeeping - output paths, logging format, dataloader workers, checkpoint file format - and do
not affect what the model learns.

In [ ]:
import importlib
import finetune as ft
importlib.reload(ft)   # pick up code changes when the notebook is re-run in the same session

train_ds, val_ds, test_ds = ft.load_split("train"), ft.load_split("val"), ft.load_split("test")
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")
print("prompt roles:", [m["role"] for m in train_ds[0]["prompt"]], "| completion roles:", [m["role"] for m in train_ds[0]["completion"]])

### Load the base model in 4-bit NF4

In [ ]:
model, tokenizer = ft.load_base()  # BitsAndBytesConfig: nf4, double quant, fp16 compute
print(ft.bnb_config())
print(f"model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB (4-bit)")

### Attach LoRA and build the trainer

In [ ]:
OUTPUT_DIR = "outputs/qlora-run"
trainer = ft.build_trainer(model, tokenizer, train_ds, val_ds, OUTPUT_DIR)
trainer.model.print_trainable_parameters()
dtypes = ft.trainable_dtypes(trainer.model)
print("trainable parameter dtypes:", dtypes)   # fp16 mixed precision needs these to be float32 only
assert set(dtypes) == {"torch.float32"}, "LoRA adapters must be float32 for fp16 training on the T4"

ex = trainer.train_dataset[0]
n_loss = sum(l != -100 for l in ex["labels"])
print(f"loss is computed on {n_loss} of {len(ex['input_ids'])} tokens (the assistant's JSON answer only)")
print("first loss tokens:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100][:30])))

### Train

In [ ]:
import time
torch.cuda.reset_peak_memory_stats()
start = time.time()
train_result = trainer.train()
print(f"training time: {(time.time() - start) / 60:.1f} min | peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

### Train and validation loss per epoch
Read from the trainer's log history (no external service needed). The validation loss must fall at every epoch.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = ft.epoch_losses(trainer.state.log_history)
loss_table = pd.DataFrame(rows).set_index("epoch")
display(loss_table.round(4))
print("validation loss decreases every epoch:", "YES" if ft.val_loss_decreasing(rows) else "NO - see note below")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(loss_table.index, loss_table["train_loss"], marker="o", color="#2a78d6", label="train")
ax.plot(loss_table.index, loss_table["val_loss"], marker="o", color="#eb6834", label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("loss (assistant tokens)"); ax.set_xticks(loss_table.index)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#ecebe7"); ax.legend(frameon=False)
ax.set_title("QLoRA training - loss per epoch", loc="left")
plt.tight_layout(); plt.savefig("outputs/loss_curve.png", dpi=130); plt.show()
loss_table.to_csv("outputs/loss_per_epoch.csv")

### Save the adapter, merge it into the base model, push to the Hub
The adapter is merged with `merge_and_unload()` into a base reloaded in **16-bit**: merging into the 4-bit weights
would bake quantization error into the released model. The result is a standalone model that loads without PEFT.

In [ ]:
import gc

ADAPTER_DIR, MERGED_DIR = "outputs/adapter", "outputs/merged"
trainer.save_model(ADAPTER_DIR)          # best epoch (load_best_model_at_end=True)
tokenizer.save_pretrained(ADAPTER_DIR)

del trainer, model                       # free the GPU before loading the 16-bit base
gc.collect(); torch.cuda.empty_cache()

merged = ft.merge_adapter(ADAPTER_DIR, MERGED_DIR)
print("merged:", type(merged).__name__, "| LoRA layers left:", any("lora" in n for n, _ in merged.named_parameters()))
print(sorted(os.listdir(MERGED_DIR)))
del merged; gc.collect()

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
HF_REPO = f"{api.whoami()['name']}/qwen2.5-1.5b-financial-news-impact"
best = min(rows, key=lambda r: r["val_loss"])
repo_page = REPO_URL.removesuffix(".git")

card = f"""---
base_model: {ft.BASE_MODEL}
library_name: transformers
license: apache-2.0
tags: [finance, qlora, peft, news-analysis, structured-output]
---
# Qwen2.5-1.5B - Financial News Impact Analysis (QLoRA, merged)

Fine-tuned from `{ft.BASE_MODEL}` with QLoRA (4-bit NF4, LoRA r={ft.LORA_R}, alpha={ft.LORA_ALPHA}, all linear layers)
on 149 synthetic financial news items (fictional events about real companies, generated by `openai/gpt-oss-120b`).
Given a headline and snippet, it returns JSON: `event_type` (11 types), `primary_entity`, `mentioned_entities`,
`impact_direction`, `magnitude`, `rationale`. Use the system prompt from the training data
(`task2_genai/news_prompts.py`, `ANALYSIS_SYSTEM`) at inference.

Best validation loss {best['val_loss']:.4f} at epoch {best['epoch']}. Code and evaluation: {repo_page} (`task2_genai`).

**Not investment advice.** Labels come from a teacher model, not market outcomes.
"""
open(f"{MERGED_DIR}/README.md", "w").write(card)

api.create_repo(HF_REPO, exist_ok=True, private=False)
api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO, commit_message="QLoRA fine-tuned, merged")
print(f"Model: https://huggingface.co/{HF_REPO}")

### Out-of-memory log
Precautions taken up front so the run fits a 16 GB T4: 4-bit weights, gradient checkpointing, paged 8-bit optimizer,
batch 4 with accumulation 4 (not batch 16), max length 1024 (not 2048), and freeing the trainer before merging.
Peak GPU memory is printed after training above.

*If an out-of-memory error occurs in your run, record it here: the error message, what was tried, and what fixed it
(typical order: batch size 4 → 2 with accumulation 4 → 8 to keep the effective batch at 16; then max length 1024 →
896, which still covers the 810-token maximum).*

---
# Task 2C - Evaluation: base vs fine-tuned

**Design.** Both models get the **identical system prompt** (`ANALYSIS_SYSTEM`), the same 18 held-out test items and
greedy decoding, so differences come from fine-tuning alone. The base model is `Qwen2.5-1.5B-Instruct` with that system
prompt and no fine-tuning; both run in fp16.

| Metric | What it measures |
|---|---|
| **ROUGE-L** (rationale; also full output) | text overlap with the reference - required by the brief |
| **BERTScore F1** (rationale) | semantic similarity with the reference, robust to paraphrase |
| **Field accuracy** | event type, direction, magnitude, primary entity - read leniently from any JSON object, so format failures are not double-counted |
| **Schema-valid %** | strict format: exactly the six keys and allowed values, nothing else |
| **Grounded %** | automatic check that every name and number in the answer appears in the input |
| **LLM-as-judge** | `qwen/qwen3.8-27b` (Groq) scores a 4-criterion rubric and a verdict, returned as JSON and validated with Pydantic |
| **Manual review** | every fine-tuned test output hand-labelled correct / partially correct / incorrect / hallucinated |

ROUGE-L partly rewards copying the teacher's phrasing, which the fine-tuned model has learned; field accuracy, grounding
and the judge measure whether the answer is actually right.

In [ ]:
import json, gc
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate as ev

test_prompts = [ex["prompt"] for ex in test_ds]
references = [json.loads(ex["completion"][0]["content"]) for ex in test_ds]
reference_raw = [ex["completion"][0]["content"] for ex in test_ds]
news = [ex["prompt"][1]["content"] for ex in test_ds]

FT_PATH = MERGED_DIR if os.path.exists(MERGED_DIR) else HF_REPO   # merged model from 2B (or the Hub copy)

def run(model_path):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16, device_map="cuda")
    outs = ev.generate(mdl, tok, test_prompts)
    del mdl; gc.collect(); torch.cuda.empty_cache()
    return outs

generations = {"base": run(ft.BASE_MODEL), "fine_tuned": run(FT_PATH)}
with open("outputs/eval_generations.jsonl", "w") as f:
    for i in range(len(test_ds)):
        f.write(json.dumps({"id": i, "news": news[i], "reference": references[i],
                            **{k: v[i] for k, v in generations.items()}}, ensure_ascii=False) + "\n")
print({k: len(v) for k, v in generations.items()}, "outputs per model")
print("\nexample - base:\n", generations["base"][0][:400], "\n\nexample - fine-tuned:\n", generations["fine_tuned"][0][:400])

### Scores

In [ ]:
judge_client = ev.groq_client()   # None if GROQ_API_KEY is not set -> judge columns are skipped
results, per_item = {}, {}
for name, raw in generations.items():
    rows = [ev.score_item(r, ref, n) for r, ref, n in zip(raw, references, news)]
    rouge_rat = ev.rouge_l([r["rationale"] for r in rows], [ref["rationale"] for ref in references])
    rouge_full = ev.rouge_l(raw, reference_raw)
    bert = ev.bertscore_f1([r["rationale"] for r in rows], [ref["rationale"] for ref in references])
    judged = [ev.judge(judge_client, n, ref, r) for n, ref, r in zip(news, references, raw)] if judge_client else None
    results[name] = ev.summarise(rows, rouge_rat, rouge_full, bert, judged)
    per_item[name] = {"rows": rows, "rouge_rat": rouge_rat, "bert": bert, "judged": judged}

summary = pd.DataFrame(results)
summary["change"] = summary["fine_tuned"] - summary["base"]
display(summary.round(3))
summary.to_csv("outputs/eval_summary.csv")
json.dump(results, open("outputs/eval_summary.json", "w"), indent=2)

**ROUGE-L comparison** (required table) and per-item wins on the identical test set:

In [ ]:
rouge_table = pd.DataFrame({
    "ROUGE-L (rationale)": [results["base"]["rougeL_rationale"], results["fine_tuned"]["rougeL_rationale"]],
    "ROUGE-L (full output)": [results["base"]["rougeL_full_output"], results["fine_tuned"]["rougeL_full_output"]],
    "BERTScore F1 (rationale)": [results["base"]["bertscore_f1_rationale"], results["fine_tuned"]["bertscore_f1_rationale"]],
}, index=["base + system prompt", "fine-tuned"])
display(rouge_table.round(4))
print("per-item ROUGE-L (rationale):", ev.paired_wins(per_item["base"]["rouge_rat"], per_item["fine_tuned"]["rouge_rat"]),
      "(b = fine-tuned, a = base)")

### Is the judge trustworthy? Calibration on known answers
Before trusting its scores, the judge is given five answers whose correct verdict is known by construction.

In [ ]:
if judge_client:
    ref0, news0 = references[0], news[0]
    first_sentence = ref0["rationale"].split(".")[0]
    controls = {
        "the reference itself -> correct": json.dumps(ref0),
        "invented figure and analyst firm -> hallucinated": json.dumps({**ref0, "rationale": first_sentence + ". Shares moved 14% after Goldman Sachs commented."}),
        "magnitude changed -> partially_correct": json.dumps({**ref0, "magnitude": "low" if ref0["magnitude"] != "low" else "high"}),
        "prose + extra key -> incorrect": "Here is my analysis:\n" + json.dumps({**ref0, "confidence": 0.9}),
        "not JSON at all -> incorrect": "The news is " + ref0["impact_direction"] + " for the company.",
    }
    calib = [{"control": k, "judge_verdict": (j.verdict if (j := ev.judge(judge_client, news0, ref0, v)) else "FAILED")}
             for k, v in controls.items()]
    display(pd.DataFrame(calib))
else:
    print("GROQ_API_KEY not set - judge skipped")

### Where fine-tuning changed the answer

In [ ]:
def labels(row):
    return "".join("✓" if row[f"{f}_correct"] else "✗" for f in ev.LABEL_FIELDS)

compare = pd.DataFrame({
    "headline": [n.split("\n")[0][len("Headline: "):][:70] for n in news],
    "base schema-valid": [r["schema_valid"] for r in per_item["base"]["rows"]],
    "ft schema-valid": [r["schema_valid"] for r in per_item["fine_tuned"]["rows"]],
    "base labels (type/dir/mag/entity)": [labels(r) for r in per_item["base"]["rows"]],
    "ft labels (type/dir/mag/entity)": [labels(r) for r in per_item["fine_tuned"]["rows"]],
    "base ROUGE-L": [round(x, 3) for x in per_item["base"]["rouge_rat"]],
    "ft ROUGE-L": [round(x, 3) for x in per_item["fine_tuned"]["rouge_rat"]],
})
display(compare)

In [ ]:
for i in range(3):   # three full side-by-side examples
    print(f"=== test item {i} ===\n{news[i]}\n\n--- reference ---\n{reference_raw[i]}")
    print(f"\n--- base ---\n{generations['base'][i]}\n\n--- fine-tuned ---\n{generations['fine_tuned'][i]}\n")

### Manual review and hallucination rate
Every fine-tuned test output is written to `outputs/manual_review.csv` with the automatic grounding issues and the
judge's verdict as aids. The reviewer reads each one against the news item and fills `manual_label` with **correct**,
**partially_correct**, **incorrect** or **hallucinated** (definitions in the README's problem statement). The labelled
file is committed to the repo as `data/manual_review.csv`, and the cell below computes the hallucination rate from it.

In [ ]:
from pathlib import Path
REVIEW_OUT, REVIEW_DONE = Path("outputs/manual_review.csv"), Path("data/manual_review.csv")
ft_judged = per_item["fine_tuned"]["judged"] or [None] * len(news)
ev.write_review_sheet(REVIEW_OUT, news, references, generations["fine_tuned"], per_item["fine_tuned"]["rows"], ft_judged)
print(f"review sheet written: {REVIEW_OUT} ({len(news)} items)")

rate = ev.hallucination_rate(REVIEW_DONE)
if rate:
    print(json.dumps(rate, indent=2))
    display(pd.read_csv(REVIEW_DONE)[["id", "manual_label", "judge_verdict", "auto_grounding_issues", "notes"]])
else:
    print("Not labelled yet: fill manual_label in the review sheet and commit it as data/manual_review.csv.")

### Qualitative analysis
*Written after reviewing the outputs above.*

**Where fine-tuning helped.** _To be written from this run's results, citing specific test items._

**Remaining failure modes and next steps.** _To be written from this run's results._

### Save the evaluation outputs (Colab)

In [ ]:
import shutil
if IN_COLAB:
    shutil.make_archive("task2_outputs", "zip", "outputs", ".")
    from google.colab import files
    files.download("task2_outputs.zip")   # loss table/curve, generations, eval summary, review sheet